# <font color="#418FDE" size="6.5" uppercase>**Embedded Sequential**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Use sequential feature selection to search forward or backward for useful feature subsets. 
- Apply embedded model-based selection using coefficients or feature importances. 
- Evaluate feature-selection stability when features are correlated or sample sizes are small. 


## **1. Sequential Search Methods**

### **1.1. Forward Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_01_01.jpg?v=1783853729" width="250">



>* Starts empty, adds best feature each step.
>* Greedy, efficient search for interpretable feature subsets.

>* Features are added based on current subset.
>* Selection builds useful combinations, not simple rankings.

>* Early choices can lock in suboptimal features.
>* Stop when gains no longer justify complexity.



### **1.2. Backward Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_01_02.jpg?v=1783853731" width="250">



>* Starts with all features, removes weakest.
>* Stops at target size or performance drop.

>* Assesses each feature’s unique value among others.
>* Removes overlap while keeping performance and simplicity.

>* Computationally costly and greedy with many features.
>* Validate carefully, especially with correlated features.



In [ ]:
#@title Python Code - Backward Selection

# Backward selection removes features one step.
# This script shows greedy subset trimming.
# We score subsets using validation accuracy.

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Set a deterministic random seed.
np.random.seed(7)

# Load a small built in dataset.
data = load_breast_cancer()
X = data.data[:, :8]
y = data.target

# Add a correlated duplicate feature.
noise = np.random.normal(0.0, 0.05, size=X.shape[0])
extra = (X[:, 0] + noise).reshape(-1, 1)
X = np.column_stack([X, extra])

# Create readable feature names.
names = list(data.feature_names[:8])
names.append("mean radius copy")

# Split data into train and validation.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, random_state=7, stratify=y
)

# Check basic shape safety first.
assert X_train.shape[1] == len(names)
assert X_train.shape[0] > X_train.shape[1]

# Define a small scoring helper.
def score_subset(columns):
    model = LogisticRegression(max_iter=1000, solver="liblinear")
    model.fit(X_train[:, columns], y_train)
    return model.score(X_valid[:, columns], y_valid)

# Start from all available features.
selected = list(range(X.shape[1]))
start_score = score_subset(selected)
steps = []

# Remove one feature at a time.
while len(selected) > 3:
    best_score = -1.0
    best_subset = None
    best_removed = None

    # Try dropping each remaining feature.
    for feature in selected:
        trial = [j for j in selected if j != feature]
        trial_score = score_subset(trial)

        # Keep the best removal.
        if trial_score > best_score:
            best_score = trial_score
            best_subset = trial
            best_removed = feature

    # Save the greedy backward step.
    steps.append((names[best_removed], len(best_subset), best_score))
    selected = best_subset

# Print a compact teaching summary.
print(f"Start: {len(names)} features, accuracy={start_score:.3f}")
for removed_name, count_left, acc in steps:
    print(f"Drop: {removed_name[:18]:18} left={count_left} acc={acc:.3f}")

# Show the final chosen subset.
final_names = [names[i] for i in selected]
print("Final subset:", ", ".join(final_names))



### **1.3. Forward Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_01_03.jpg?v=1783853733" width="250">



>* Starts empty, adding best feature each step.
>* Stops when gains fade, avoiding exhaustive search.

>* Features are added in informative stages.
>* Value depends on current selected features.

>* Greedy choices can miss better subsets.
>* Use validation and stopping rules carefully.



## **2. Embedded Model Selection**

### **2.1. Coefficient Based Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_02_01.jpg?v=1783853726" width="250">



>* Models use coefficients to judge feature influence.
>* Near-zero coefficients suggest less useful features.

>* Regularization shrinks weak features toward zero.
>* Standardize features before comparing coefficient sizes.

>* Coefficients can be unstable with correlated features.
>* Check importance across resamples and context.



### **2.2. Model Based Importance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_02_02.jpg?v=1783853724" width="250">



>* Importance reflects a model’s predictive reliance.
>* Captures interactions, nonlinear effects, and split gains.

>* Finds predictors through interactions and combined effects.
>* Importance aids prediction, not causation or universality.

>* Keep strong features, remove weak ones.
>* Validate importance stability across samples.



### **2.3. Feature Importance Stability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_02_03.jpg?v=1783853727" width="250">



>* Importance should stay consistent across samples.
>* Stability prevents overconfident feature interpretations.

>* Correlated features can swap importance unpredictably.
>* Small samples make importance rankings less reliable.

>* Trust features stable across resampling and retraining.
>* Unstable importance requires cautious, grouped interpretation.



In [ ]:
#@title Python Code - Feature Importance Stability

# This script shows unstable feature importance.
# Correlated inputs can swap importance ranks.
# Small samples make rankings less reliable.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Build correlated features and target.
n_samples = 80
x1 = rng.normal(size=n_samples)
noise = rng.normal(scale=0.15, size=n_samples)

# Create a near-duplicate predictor.
x2 = x1 + noise
x3 = rng.normal(size=n_samples)
y = 3 * x1 + rng.normal(scale=0.8, size=n_samples)

# Combine features into one matrix.
X = np.column_stack([x1, x2, x3])
feature_names = ["x1", "x2", "x3"]

# Check shapes before fitting models.
assert X.shape == (n_samples, 3)
assert y.shape == (n_samples,)

# Store coefficients from repeated resamples.
coef_rows = []

# Refit many times on small samples.
for seed in range(30):
    sample_idx = np.random.default_rng(seed).choice(
        n_samples, size=35, replace=False
    )

    # Fit linear regression with numpy.
    X_sub = X[sample_idx]
    y_sub = y[sample_idx]
    X_design = np.column_stack([np.ones(len(X_sub)), X_sub])

    # Solve least squares safely.
    beta, _, _, _ = np.linalg.lstsq(
        X_design, y_sub, rcond=None
    )
    coef_rows.append(beta[1:])

# Summarize coefficient stability.
coef_df = pd.DataFrame(coef_rows, columns=feature_names)
mean_abs = coef_df.abs().mean().sort_values(ascending=False)
std_abs = coef_df.abs().std().reindex(mean_abs.index)

# Count how often each feature ranks first.
rank_winner = coef_df.abs().idxmax(axis=1).value_counts()
rank_winner = rank_winner.reindex(feature_names, fill_value=0)

# Print a compact interpretation.
print("Average absolute coefficients:")
print(mean_abs.round(2).to_string())
print("\nStandard deviation across resamples:")
print(std_abs.round(2).to_string())
print("\nTimes ranked most important:")
print(rank_winner.to_string())

# Plot coefficient spread across resamples.
coef_df.abs()[mean_abs.index].boxplot()
plt.title("Importance stability across resamples")
plt.ylabel("Absolute coefficient")
plt.tight_layout()
plt.show()



## **3. Selection Stability**

### **3.1. Stability Pitfalls**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_03_01.jpg?v=1783853735" width="250">



>* Selected features may change with small perturbations.
>* Unstable subsets can mislead interpretation and decisions.

>* Similar accuracy can hide unstable feature choices.
>* Instability misleads interpretation and real decisions.

>* Small subsets can still be unstable.
>* Judge selection by consistency, not size.



### **3.2. Correlated Feature Stability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_03_02.jpg?v=1783853737" width="250">



>* Correlated features can swap during selection.
>* Different choices may reflect the same signal.

>* Correlated features make selection results unstable.
>* Methods may credit one substitute predictor.

>* Judge stability by retained signal, not names.
>* Balance prediction needs with interpretation goals.



In [ ]:
#@title Python Code - Correlated Feature Stability

# Correlated features can swap during selection.
# Small samples often increase selection instability.
# This script measures group level stability.

import numpy as np
import pandas as pd

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Build correlated features around one signal.
n_samples = 80
base_signal = rng.normal(size=n_samples)

# Create interchangeable correlated predictors.
x_income_a = base_signal + rng.normal(0, 0.25, n_samples)
x_income_b = base_signal + rng.normal(0, 0.25, n_samples)

# Add one independent useful predictor.
x_age = rng.normal(size=n_samples)
y = 2 * base_signal + 1.5 * x_age

y = y + rng.normal(0, 0.6, n_samples)

# Assemble a small feature table.
X = pd.DataFrame({
    "income_proxy_a": x_income_a,
    "income_proxy_b": x_income_b,
    "age": x_age,
    "noise": rng.normal(size=n_samples),
})

# Check the expected table shape.
assert X.shape == (80, 4)

# Define a simple forward selection helper.
def forward_select(frame, target, choose=2):
    chosen = []

    # Add one best feature each step.
    for _ in range(choose):
        scores = {}

        # Test every remaining candidate.
        for name in frame.columns:
            if name in chosen:
                continue

            # Fit least squares and score fit.
            cols = chosen + [name]
            design = np.column_stack([
                np.ones(len(frame)), frame[cols].to_numpy()
            ])
            coef, _, _, _ = np.linalg.lstsq(
                design, target, rcond=None
            )
            pred = design @ coef
            sse = np.sum((target - pred) ** 2)
            scores[name] = -sse

        # Keep the best current feature.
        best_name = max(scores, key=scores.get)
        chosen.append(best_name)

    return chosen

# Repeat selection on many small resamples.
counts = {name: 0 for name in X.columns}
group_hits = 0

# Track whether either proxy appears.
for _ in range(40):
    rows = rng.choice(n_samples, size=35, replace=False)
    picked = forward_select(X.iloc[rows], y[rows], choose=2)

    # Update feature and group counts.
    for name in picked:
        counts[name] += 1
    if any(name.startswith("income_proxy") for name in picked):
        group_hits += 1

# Summarize instability across correlated names.
proxy_total = counts["income_proxy_a"] + counts["income_proxy_b"]
print("Feature counts across 40 resamples:")
print(f"income_proxy_a: {counts['income_proxy_a']}")
print(f"income_proxy_b: {counts['income_proxy_b']}")
print(f"age: {counts['age']}")
print(f"noise: {counts['noise']}")
print(f"Either income proxy selected: {group_hits}/40")
print(f"Total proxy selections: {proxy_total}")
print("Interpretation: names vary, shared signal stays.")



### **3.3. Small Sample Stability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_03_03.jpg?v=1783853738" width="250">



>* Small datasets make feature choices unstable.
>* Seek reproducible predictors, not sample-specific noise.

>* Small samples can select accidental patterns.
>* Check stability alongside predictive performance.

>* Check features across resamples for consistency.
>* Use uncertainty, simplicity, and domain knowledge.



In [ ]:
#@title Python Code - Small Sample Stability

# Small samples can change chosen features easily.
# This script measures selection stability simply.
# We compare repeated tiny resamples carefully.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectFromModel

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(7)
np.random.seed(7)

# Build correlated data with few samples.
X, y = make_classification(
    n_samples=60, n_features=8, n_informative=3,
    n_redundant=3, n_repeated=0, random_state=7
)

# Name features for readable output.
feature_names = np.array([
    "f0", "f1", "f2", "f3",
    "f4", "f5", "f6", "f7"
])

# Check basic shape safety first.
assert X.shape == (60, 8)
assert y.shape == (60,)

# Import model only when needed.
from sklearn.linear_model import LogisticRegression

# Store how often each feature appears.
counts = np.zeros(X.shape[1], dtype=int)
selected_sets = []

# Repeat selection on many tiny resamples.
for _ in range(30):
    idx = rng.choice(len(y), size=24, replace=False)
    X_small, y_small = X[idx], y[idx]

    # Fit a simple embedded selector.
    model = LogisticRegression(
        penalty="l1", solver="liblinear", C=0.25,
        random_state=7, max_iter=1000
    )

    # Select nonzero coefficient features.
    selector = SelectFromModel(model, threshold=1e-8)
    selector.fit(X_small, y_small)

    # Save selected feature names.
    mask = selector.get_support()
    counts += mask.astype(int)
    selected_sets.append(tuple(feature_names[mask]))

# Summarize repeated selection stability.
freq = counts / len(selected_sets)
order = np.argsort(freq)[::-1]

# Count unique selected subsets.
unique_subsets = len(set(selected_sets))
most_stable = feature_names[order[:3]]

# Print a short interpretation.
print("Repeated resamples:", len(selected_sets))
print("Sample size each run:", 24)
print("Unique selected subsets:", unique_subsets)
print("Most often selected:", ", ".join(most_stable))
print("Top selection rates:", np.round(freq[order[:5]], 2))
print("Higher variation means lower stability.")

# Plot selection frequency by feature.
plt.figure(figsize=(7, 3.5))
plt.bar(feature_names[order], freq[order], color="steelblue")
plt.ylim(0, 1)
plt.ylabel("Selection frequency")
plt.xlabel("Feature")

# Add a helpful title.
plt.title("Small sample stability across repeated resamples")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Embedded Sequential**</font>


In this lecture, you learned to:
- Use sequential feature selection to search forward or backward for useful feature subsets. 
- Apply embedded model-based selection using coefficients or feature importances. 
- Evaluate feature-selection stability when features are correlated or sample sizes are small. 

In the next Module (Module 21), we will go over 'None'